<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

<h1 style="text-align: center;">02 Data Communication</h1>
<hr>


## **8. Radio Spectrum**

<div class="alert alert-light">

  **Name:** (add your name here)
  
  **Matriculation Number:** (add your number here)
  
</div>




### 8.1 Introduction: From Baseband to Shared Radio Spectrum

A wireless link must solve two problems: turn data into a signal that can be transmitted efficiently, and place that signal in a frequency band where it can coexist with other transmissions. Choosing a modulation scheme therefore involves both communication performance and the use of a shared resource: radio spectrum.

A **baseband signal** represents the information before it is shifted to a radio-frequency carrier. Its spectrum is located around zero frequency rather than around a higher carrier frequency. Electrical baseband signals can be transmitted directly over suitable wired links. For a conventional mobile radio link, however, feeding a low-frequency baseband waveform directly into a small antenna is generally impractical. Efficient antennas are typically sized in relation to wavelength, $\lambda=c/f$, so low frequencies imply large antenna dimensions; much smaller antennas face efficiency and bandwidth constraints. Any constant, or DC, component also cannot propagate as a radiated wave carrying changing information.

Modulation and frequency conversion place the data-bearing signal around a **carrier frequency** suited to the antenna, propagation environment, and permitted operating band. Separate links can then occupy different frequency channels, and a receiver can use filtering to select the desired channel. Carrier selection establishes where the signal lies in the spectrum; the symbol rate and pulse shape largely determine how wide that signal is.

Moving to a carrier does not create unlimited room for transmissions. Mobile networks, broadcasting, navigation, and other services share frequency bands with different propagation properties and equipment requirements. Signals received in overlapping bands can interfere, so spectrum use must be coordinated. National authorities manage frequency use within an international framework coordinated through the International Telecommunication Union (ITU). The purpose includes efficient sharing of usable spectrum and limiting harmful interference, as explained in the [ITU handbook on national spectrum management](https://www.itu.int/dms_pub/itu-r/opb/hdb/R-HDB-21-2005-R1-PDF-E.pdf).

**Spectral efficiency** describes the data rate carried per unit of bandwidth, usually in bit/s/Hz. For a fixed data rate, using less bandwidth can leave more room for other links; for a fixed bandwidth, higher spectral efficiency can support more data. These benefits depend on controlling unwanted emissions and maintaining reliable reception. Higher-order modulation carries more bits per symbol, but typically requires a cleaner received signal when average power is held fixed.

The experiments first illustrate 16-QAM and 256-QAM in time, then compare QPSK and 256-QAM in frequency at the same bit rate. They show how carrier frequency and symbol rate affect the transmitted spectrum. This provides the technical basis for the discussion of spectral efficiency and spectrum regulation in Section 8.5.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import random

Data is the information to be conveyed, such as a sequence of bits representing text or measurements. A signal is its physical representation, such as a changing voltage or electromagnetic field. Calling the information digital does not make the transmitted radio wave a sequence of abstract zeros and ones: the receiver must infer those bits from a measured waveform.

For QAM, groups of bits select complex-valued symbols with in-phase and quadrature components. These components describe the baseband waveform. Combining them with cosine and sine carriers produces the real-valued waveform used for passband transmission. The examples use low carrier frequencies so that the oscillations remain easy to inspect; practical radio systems place the corresponding signals in their operating frequency bands.


#### 8.1.1 Digital Data

The following cell generates example bit sequences. The 512-bit sequence is used for both QAM schemes so that the later comparison carries the same amount of information.


In [ ]:
data = random.choices([0, 1], k=532)
data_long = random.choices([0, 1], k=512)

#### 8.1.2 Illustrative Carrier Signals

The following cell creates two unmodulated sinusoids at 200 Hz and 1000 Hz. These are illustrative carriers, rather than data-bearing baseband waveforms. Their different frequencies make it possible to compare where the modulated signals appear in the spectrum. The time plot shows a short interval so that the individual oscillations are visible.


In [ ]:
sampling_rate = 10000  # Sampling rate in samples per second
duration = 1  # Duration in seconds
freq_a_Hertz = 200
freq_b_Hertz = 1000

t_min_sec = 0
t_max_sec = t_min_sec + duration
number_of_samples = sampling_rate * (t_max_sec-t_min_sec)

sampling_interval_sec = (t_max_sec-t_min_sec)/number_of_samples

# Time vector
time_sec = np.linspace(0, duration, int(
    sampling_rate * duration), endpoint=False)

# Harmonic signal
signal_a = np.sin(2 * np.pi * freq_a_Hertz * time_sec)
signal_b = np.sin(2 * np.pi * freq_b_Hertz * time_sec)

# add leading and trailing zeros to the signals
time_sec = np.concatenate(
    (np.linspace(t_min_sec - (t_max_sec-t_min_sec), t_min_sec, round(number_of_samples)), time_sec, np.linspace(t_max_sec, t_max_sec+(t_max_sec-t_min_sec), round(number_of_samples))))
signal_a = np.concatenate(
    (np.array(np.zeros(len(signal_a))), signal_a, np.array(np.zeros(len(signal_a)))))
signal_b = np.concatenate(
    (np.array(np.zeros(len(signal_b))), signal_b, np.array(np.zeros(len(signal_b)))))

**Plot**

In [ ]:
fig, ax_time = plt.subplots(figsize=(8, 4))
theLineWidth = 3
ax_time.cla()
ax_time.grid(color="#AAAAAA", linewidth=.5, linestyle='-', alpha=1)
ax_time.set_ylabel('signals in time domain, amplitude')
ax_time.set_xlabel('time (s)')
ax_time.axis([None, None, None, None])

ax_time.plot(time_sec, signal_a, label=[
    'Unmodulated baseband signal a (' + str(freq_a_Hertz) + ' Hz)'], color='m', linewidth=theLineWidth)
ax_time.plot(time_sec, signal_b, label=[
    'Unmodulated baseband signal b (' + str(freq_b_Hertz) + ' Hz)'], color='w', linewidth=1)

ax_time.set_xlim(0.1, 0.20)
ax_time.set_facecolor((.2, 0.2, 0.3))
plt.legend(fancybox=True, framealpha=1,
           loc="upper left", facecolor=(.6, 0.61, 0.8))
plt.tight_layout()

### 8.2 Quadrature Amplitude Modulation (QAM)

QAM represents each group of bits with a choice of in-phase and quadrature components, which together determine the carrier's amplitude and phase. The following examples compare 16-QAM and 256-QAM, focusing on how the number of bits per symbol affects the symbol rate needed to carry a message.


#### 8.2.1 16-QAM Modulation

**16-QAM** uses sixteen constellation points arranged in a four-by-four grid. Each point represents four bits: two select the in-phase (I) level and two select the quadrature (Q) level. The selected coordinates determine the amplitude and phase of the transmitted carrier.

The plots below compare the ideal constellation with one disturbed by noise. A receiver estimates which point was transmitted from the measured I and Q values. Noise moves a received point away from its ideal location; a sufficiently large displacement can cause an incorrect decision.


In [ ]:
def generate_16qam_constellation():
    # Create the real and imaginary parts of the 16-QAM constellation
    real_part = np.array([-3, -1, 1, 3])
    imag_part = np.array([-3, -1, 1, 3])
    
    # Create the full constellation grid by combining real and imaginary parts
    constellation = np.array([complex(r, i) for r in real_part for i in imag_part])
    
    return constellation

def add_noise(constellation, noise_variance):
    noise = np.sqrt(noise_variance / 2) * (np.random.randn(len(constellation)) + 1j * np.random.randn(len(constellation)))
    noisy_constellation = constellation + noise
    return noisy_constellation

def plot_constellation(ax, constellation, title, color='blue'):
    ax.cla()
    ax.axis([None, None, None, None])
    ax.scatter(constellation.real, constellation.imag, color=color, s=30, label=title)
    ax.legend(fancybox=True, framealpha=1,
               loc="upper right", facecolor=(.6, 0.61, 0.8))
    ax.axhline(0, color='white', lw=2)
    ax.axvline(0, color='white', lw=2)
    ax.grid(True)
    ax.set_xlabel('In-phase (I)')
    ax.set_ylabel('Quadrature (Q)')
    ax.set_xlim(-5, 5)
    ax.set_ylim(-5, 5)
    ax.set_facecolor((.2, 0.2, 0.3))

if __name__ == "__main__":
    constellation = generate_16qam_constellation()
    noise_variance = 0.3  # Adjust this value to control noise level
    noisy_constellation = add_noise(constellation, noise_variance)
    
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    theLineWidth = 2
    fig.set_facecolor((.6, 0.61, 0.8))

    plot_constellation(axes[1], noisy_constellation, "16-QAM (with noise)", color='red')
    plot_constellation(axes[0], constellation, "16-QAM", color='blue')    
    plt.tight_layout()


#### 8.2.2 256-QAM Modulation

**256-QAM** extends the constellation to a sixteen-by-sixteen grid. Each point carries eight bits, with four selecting the I level and four selecting the Q level. At the same symbol rate, it carries twice as many bits as 16-QAM. Alternatively, it can carry the same bit rate using half the symbol rate.

More states must be distinguished at the receiver. When constellations are normalized to the same average power, 256-QAM points lie closer together than 16-QAM points. A higher signal-to-noise ratio (SNR) is therefore needed to maintain comparable detection reliability.

The following table summarizes the comparison. The noise-tolerance comparison assumes equal average signal power and otherwise comparable receiver and channel conditions.

| Property | 16-QAM | 256-QAM |
|---|---|---|
| Constellation size | 16 points in a 4 × 4 grid | 256 points in a 16 × 16 grid |
| Bits per symbol | 4 | 8 |
| Levels on each I/Q axis | 4 | 16 |
| Bits assigned to each I/Q axis | 2 | 4 |
| Symbol rate at 400 bit/s | 100 symbols/s | 50 symbols/s |
| Symbols needed for 512 bits | 128 | 64 |
| Transmission duration at 400 bit/s | 1.28 s | 1.28 s |
| Neighbouring-point separation at equal average power | Larger | Smaller |
| Noise tolerance at equal average power | Greater | Lower |
| SNR needed for comparable detection reliability | Lower | Higher |

The constellation plots in this section use unnormalized integer coordinates and the same added-noise variance. They illustrate the arrangements of the points, but are not a comparison at equal signal power or equal SNR. The spectrum experiment in Section 8.4 normalizes the constellations before comparing their bandwidths.


In [ ]:
def generate_256qam_constellation():
    real_part = np.arange(-15, 16, 2)  # Real part with 16 levels
    imag_part = np.arange(-15, 16, 2)  # Imaginary part with 16 levels    
    constellation = np.array([complex(r, i) for r in real_part for i in imag_part])
    return constellation

def add_noise(constellation, noise_variance):
    noise = np.sqrt(noise_variance / 2) * (np.random.randn(len(constellation)) + 1j * np.random.randn(len(constellation)))
    noisy_constellation = constellation + noise
    return noisy_constellation

def plot_constellation(ax, constellation, title, color='blue'):
    ax.cla()
    ax.axis([None, None, None, None])
    ax.scatter(constellation.real, constellation.imag, color=color, s=30, label=title)
    ax.legend(fancybox=True, framealpha=1,
               loc="upper right", facecolor=(.6, 0.61, 0.8))
    ax.axhline(0, color='white', lw=2)
    ax.axvline(0, color='white', lw=2)
    ax.grid(True)
    ax.set_xlabel('In-phase (I)')
    ax.set_ylabel('Quadrature (Q)')
    ax.set_facecolor((.2, 0.2, 0.3))
    ax.set_xlim(-17, 17)
    ax.set_ylim(-17, 17)

if __name__ == "__main__":
    constellation = generate_256qam_constellation()
    noise_variance = 0.3  # Adjust this value to control noise level
    noisy_constellation = add_noise(constellation, noise_variance)
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    theLineWidth = 2
    fig.set_facecolor((.6, 0.61, 0.8))
    plot_constellation(axes[1], noisy_constellation, "256-QAM (with noise)", color='red')
    plot_constellation(axes[0], constellation, "256-QAM", color='blue')    
    plt.tight_layout()


### 8.3 Modulation and Time-Domain Comparison

The next cells map the same 512-bit message to 16-QAM and 256-QAM symbols, then generate waveforms at the two illustrative carrier frequencies. A 16-QAM symbol carries four bits, so the message needs 128 symbols. A 256-QAM symbol carries eight bits, so it needs only 64 symbols.

The symbol rates are set to 100 symbols/s for 16-QAM and 50 symbols/s for 256-QAM. Both therefore transmit at 400 bit/s and take 1.28 s to send the message. This keeps the amount of data, bit rate, and total duration fixed while changing how often the carrier's amplitude and phase can change.

The plots show only the first 0.2 s. Compare the timing of symbol changes within each carrier-frequency pair, and distinguish those changes from the faster carrier oscillations. The waveforms are not normalized to equal power here, so their heights should not be used to compare energy efficiency. Section 8.4 moves to the frequency domain and uses QPSK versus 256-QAM to make the bandwidth difference more pronounced.


In [ ]:

def bits_to_256qam(bits):
    # Ensure the number of bits is a multiple of 8
    if len(bits) % 8 != 0:
        raise ValueError("The number of bits must be a multiple of 16 for 256-QAM.")
    
    constellation = generate_256qam_constellation()
    
    symbols = []
    for i in range(0, len(bits), 8):
        bit_group = bits[i:i+8]
        # Convert the 8-bit group to an integer (0-255)
        symbol_index = int("".join(map(str, bit_group)), 2)
        # Map to corresponding 256-QAM symbol
        symbols.append(constellation[symbol_index])
    
    return np.array(symbols)


def bits_to_16qam(bits):
    # Ensure the number of bits is a multiple of 4
    if len(bits) % 4 != 0:
        raise ValueError("The number of bits must be a multiple of 4 for 16-QAM.")
    
    constellation = generate_16qam_constellation()
    
    symbols = []
    for i in range(0, len(bits), 4):
        bit_group = bits[i:i+4]
        # Convert the 4-bit group to an integer (0-15)
        symbol_index = int("".join(map(str, bit_group)), 2)
        # Map to corresponding 16-QAM symbol
        symbols.append(constellation[symbol_index])
    
    return np.array(symbols)

# Function to map 16-QAM symbols back to bits
def convert_16qam_to_bits(symbols):
    constellation = generate_16qam_constellation()
    bits = []
    
    for symbol in symbols:
        # Find the closest constellation point
        distances = np.abs(constellation - symbol)
        closest_index = np.argmin(distances)
        # Convert the symbol index back to bits
        bits.append(np.binary_repr(closest_index, width=4))
    
    # Flatten the bits
    return [int(b) for bit_group in bits for b in bit_group]

# Function to map 256-QAM symbols back to bits
def convert_256qam_to_bits(symbols):
    constellation = generate_256qam_constellation()
    bits = []
    
    for symbol in symbols:
        # Find the closest constellation point
        distances = np.abs(constellation - symbol)
        closest_index = np.argmin(distances)
        # Convert the symbol index back to bits
        bits.append(np.binary_repr(closest_index, width=8))
    
    # Flatten the bits
    return [int(b) for bit_group in bits for b in bit_group]

symbols_16qam = bits_to_16qam(data_long)
symbols_256qam = bits_to_256qam(data_long)
m_16QAM = int(len(data_long)/len(symbols_16qam));
m_256QAM = int(len(data_long)/len(symbols_256qam));

print ("16-QAM: ", len(symbols_16qam), "symbols (", m_16QAM, "bits per symbol )")
print ("256-QAM:", len(symbols_256qam), "symbols (", m_256QAM, "bits per symbol )")
 

In [ ]:

symbol_rate_16QAM = 100  # symbols per second
symbol_rate_256QAM = 50  # symbols per second

t_16QAM = np.linspace(0, 1/symbol_rate_16QAM, int(sampling_rate/symbol_rate_16QAM), endpoint=False) # Time vector for one symbol duration
t_256QAM = np.linspace(0, 1/symbol_rate_256QAM, int(sampling_rate/symbol_rate_256QAM), endpoint=False) # Time vector for one symbol duration

signal_a_16QAM_modulated = np.array([])
signal_b_16QAM_modulated = np.array([])
number_of_16QAM_symbols = 0
for  symbol in symbols_16qam:
    number_of_16QAM_symbols = number_of_16QAM_symbols + 1
    next_symbol_a = np.cos(2 * np.pi * freq_a_Hertz * t_16QAM) * symbol.real - np.sin(2 * np.pi * freq_a_Hertz * t_16QAM) * symbol.imag;
    next_symbol_b = np.cos(2 * np.pi * freq_b_Hertz * t_16QAM) * symbol.real - np.sin(2 * np.pi * freq_b_Hertz * t_16QAM) * symbol.imag;
    signal_a_16QAM_modulated = np.concatenate((signal_a_16QAM_modulated, next_symbol_a))
    signal_b_16QAM_modulated = np.concatenate((signal_b_16QAM_modulated, next_symbol_b))


signal_a_256QAM_modulated = np.array([])
signal_b_256QAM_modulated = np.array([])
number_of_256QAM_symbols = 0
for  symbol in symbols_256qam:
    number_of_256QAM_symbols = number_of_256QAM_symbols + 1
    next_symbol_a = np.cos(2 * np.pi * freq_a_Hertz * t_256QAM) * symbol.real - np.sin(2 * np.pi * freq_a_Hertz * t_256QAM) * symbol.imag;
    next_symbol_b = np.cos(2 * np.pi * freq_b_Hertz * t_256QAM) * symbol.real - np.sin(2 * np.pi * freq_b_Hertz * t_256QAM) * symbol.imag;
    signal_a_256QAM_modulated = np.concatenate((signal_a_256QAM_modulated, next_symbol_a))
    signal_b_256QAM_modulated = np.concatenate((signal_b_256QAM_modulated, next_symbol_b))

**Plot 16-QAM**

In [ ]:
fig, ax_time = plt.subplots(figsize=(8, 4))
theLineWidth = 1.5
ax_time.cla()
ax_time.grid(color="#AAAAAA", linewidth=.5, linestyle='-', alpha=1)
ax_time.set_ylabel('modulated signal in time domain, amplitude')
ax_time.set_xlabel('time (s)')
ax_time.axis([None, None, None, None])

ax_time.plot(np.linspace(0, len(signal_a_16QAM_modulated)*sampling_interval_sec, len(signal_a_16QAM_modulated)), signal_a_16QAM_modulated, label=[
    '16QAM-modulated signal a (' + str(freq_a_Hertz) + ' Hz)'], color='m', linewidth=theLineWidth)
ax_time.set_facecolor((.2, 0.2, 0.3))
plt.legend(fancybox=True, framealpha=1,
           loc="upper left", facecolor=(.6, 0.61, 0.8))

texstring = str(len(data_long))+" bits in " + str(number_of_16QAM_symbols) + " symbols\nsymbol rate set to: " + str(symbol_rate_16QAM) + " symbols/s\nresulting bit rate:" + str(symbol_rate_16QAM*m_16QAM) + " b/s\nresulting total duration:" + str(len(data_long) / (symbol_rate_16QAM*m_16QAM)) + "s"
props = dict(boxstyle='round', alpha=1, facecolor=(.6, 0.61, 0.8), edgecolor='w')
tex = ax_time.text(0.005, 0.0, texstring , verticalalignment='center', fontsize=9)
tex.set_bbox(props)
ax_time.set_xlim(0, 0.2)
plt.tight_layout()
plt.grid(True)


fig, ax_time = plt.subplots(figsize=(8, 4))
theLineWidth = 1.5
ax_time.cla()
ax_time.grid(color="#AAAAAA", linewidth=.5, linestyle='-', alpha=1)
ax_time.set_ylabel('modulated signal in time domain, amplitude')
ax_time.set_xlabel('time (s)')
ax_time.axis([None, None, None, None])

ax_time.plot(np.linspace(0, len(signal_b_16QAM_modulated)*sampling_interval_sec, len(signal_b_16QAM_modulated)), signal_b_16QAM_modulated, label=[
    '16QAM-modulated signal b (' + str(freq_b_Hertz) + ' Hz)'], color='w', linewidth=theLineWidth)
ax_time.set_facecolor((.2, 0.2, 0.3))
plt.legend(fancybox=True, framealpha=1,
           loc="upper left", facecolor=(.6, 0.61, 0.8))

tex = ax_time.text(0.005, 0.0, texstring , verticalalignment='center', fontsize=9)
tex.set_bbox(props)
ax_time.set_xlim(0.0, 0.2)
plt.tight_layout()
plt.grid(True)

**Plot 256-QAM**

In [ ]:
fig, ax_time = plt.subplots(figsize=(8, 4))
theLineWidth = 1.5
ax_time.cla()
ax_time.grid(color="#AAAAAA", linewidth=.5, linestyle='-', alpha=1)
ax_time.set_ylabel('modulated signal in time domain, amplitude')
ax_time.set_xlabel('time (s)')
ax_time.axis([None, None, None, None])

ax_time.plot(np.linspace(0, len(signal_a_256QAM_modulated)*sampling_interval_sec, len(signal_a_256QAM_modulated)), signal_a_256QAM_modulated, label=[
    '256QAM-modulated signal a (' + str(freq_a_Hertz) + ' Hz)'], color='b', linewidth=theLineWidth)
ax_time.set_facecolor((.2, 0.2, 0.3))
plt.legend(fancybox=True, framealpha=1,
           loc="upper left", facecolor=(.6, 0.61, 0.8))

texstring = str(len(data_long))+" bits in " + str(number_of_256QAM_symbols) + " symbols\nsymbol rate set to: " + str(symbol_rate_256QAM) + " symbols/s\nresulting bit rate:" + str(symbol_rate_256QAM*m_256QAM) + " b/s\nresulting total duration:" + str(len(data_long) / (symbol_rate_256QAM*m_256QAM) ) + "s"
props = dict(boxstyle='round', alpha=1, facecolor=(.6, 0.61, 0.8), edgecolor='w')
tex = ax_time.text(0.005, 0.0, texstring , verticalalignment='center', fontsize=9)
tex.set_bbox(props)
ax_time.set_xlim(0, 0.2)
plt.tight_layout()
plt.grid(True)

fig, ax_time = plt.subplots(figsize=(8, 4))
theLineWidth = 1.5
ax_time.cla()
ax_time.grid(color="#AAAAAA", linewidth=.5, linestyle='-', alpha=1)
ax_time.set_ylabel('modulated signal in time domain, amplitude')
ax_time.set_xlabel('time (s)')
ax_time.axis([None, None, None, None])

ax_time.plot(np.linspace(0, len(signal_b_256QAM_modulated)*sampling_interval_sec, len(signal_b_256QAM_modulated)), signal_b_256QAM_modulated, label=[
    '256QAM-modulated signal b (' + str(freq_b_Hertz) + ' Hz)'], color='g', linewidth=theLineWidth)
ax_time.set_facecolor((.2, 0.2, 0.3))
plt.legend(fancybox=True, framealpha=1,
           loc="upper left", facecolor=(.6, 0.61, 0.8))
tex = ax_time.text(0.005, 0.0, texstring , verticalalignment='center', fontsize=9)
tex.set_bbox(props)
ax_time.set_xlim(0, 0.2)
plt.tight_layout()
plt.grid(True)

### 8.4 Radio Spectrum: How Modulation Affects Signal Bandwidth

A radio transmission must fit within a usable frequency band while carrying the required data rate. This creates a design problem: a data-bearing signal occupies a range of frequencies, so specifying its carrier frequency alone is not enough to separate it from neighbouring transmissions. If their spectra overlap, one transmission can interfere with another. The practical question is therefore how much bandwidth a message requires, and which signal parameters determine that bandwidth.

An ideal sinusoid of unlimited duration has a single positive-frequency component. Transmitting data changes its amplitude and phase over time, introducing frequency components around the carrier. Faster symbol changes generally require a wider band when the pulse shape is kept the same. The symbol rate therefore plays a central role in the signal's bandwidth. Changing the carrier frequency moves the spectrum to a different position; it does not by itself increase the number of bits transmitted per second.

The following experiment compares **QPSK (4-QAM) and 256-QAM at the same bit rate**. The relationship between bit rate $R_b$, symbol rate $R_s$, and constellation size $M$ is

$$
R_b = R_s\log_2(M).
$$

QPSK carries two bits per symbol at 200 symbols/s, giving 400 bit/s. The 256-QAM signal carries eight bits per symbol at 50 symbols/s, also giving 400 bit/s. This produces a stronger bandwidth contrast than the earlier 16-QAM comparison. The spectrum experiment uses a longer shared message of 8192 bits, lasting 20.48 s, so that many analysis windows can be averaged. The earlier time-domain examples remain short enough to inspect easily.

A single FFT of random data produces an irregular spectrum whose peaks can hide its overall shape. The code instead estimates the **power spectral density (PSD)** with [Welch's method](https://docs.scipy.org/doc/scipy-1.13.0/reference/generated/scipy.signal.welch.html), averaging the spectra of overlapping windows. Longer signals provide more windows for averaging; length alone does not narrow the transmitted bandwidth when the symbol rate stays fixed. The constellations have equal average symbol power, and each plotted spectrum is normalized to its own peak to emphasize width rather than height.

The upper plot retains the overview at 200 Hz and 1000 Hz. The two lower plots zoom into the carriers using the same frequency span and vertical scale. Blue represents QPSK and orange represents 256-QAM. Thin lines show the averaged spectra; stronger dashed curves show the expected rectangular-pulse envelope around each positive carrier. Finite-record averaging and the contribution of the negative-frequency image can cause the measured spectrum to differ from this idealized envelope.

For rectangular symbol pulses, this envelope follows $\mathrm{sinc}^2((f-f_c)/R_s)$, where $\mathrm{sinc}(x)=\sin(\pi x)/(\pi x)$. Its first zeros lie at $f_c-R_s$ and $f_c+R_s$, so the **main-lobe width between the first zeros** is $2R_s$. The arrows therefore mark 400 Hz for QPSK and 100 Hz for 256-QAM: a fourfold difference at the same bit rate. This is an explicit bandwidth comparison criterion, not a claim that all transmitted power lies within those limits.

Abrupt symbol changes produce sidelobes outside the main lobe. Practical transmitters use pulse shaping and filtering to control those emissions. At the same symbol rate and pulse shape, increasing QAM order would increase bit rate without inherently narrowing the spectrum; the narrower main lobe here results from the lower symbol rate used to maintain 400 bit/s.

Higher-order QAM also places more constellation points within a given signal-power budget, making them harder to distinguish in noise. The design challenge is to meet the data-rate requirement within the available bandwidth while maintaining reliable detection. Section 8.5 draws the broader conclusions for spectral efficiency and spectrum use.


In [ ]:
from scipy.signal import welch

# A longer shared message provides many independent windows for spectral averaging.
spectrum_bit_rate = 400  # bit/s for both modulation schemes
spectrum_bits = np.random.default_rng(8).integers(0, 2, 8192)
spectrum_fs = sampling_rate
spectrum_carriers = (freq_a_Hertz, freq_b_Hertz)
spectrum_schemes = [(4, "QPSK (4-QAM)", "tab:blue"),
                    (256, "256-QAM", "tab:orange")]

def spectrum_qam_symbols(bits, order):
    bits_per_symbol = int(np.log2(order))
    groups = bits.reshape(-1, bits_per_symbol)
    indices = groups @ (2 ** np.arange(bits_per_symbol - 1, -1, -1))
    side = int(np.sqrt(order))
    levels = np.arange(-(side - 1), side, 2)
    constellation = (levels[:, None] + 1j * levels[None, :]).ravel()
    constellation /= np.sqrt(np.mean(np.abs(constellation) ** 2))
    return constellation[indices]

spectrum_results = []
for order, label, color in spectrum_schemes:
    symbol_rate = spectrum_bit_rate / np.log2(order)
    samples_per_symbol = int(round(spectrum_fs / symbol_rate))
    symbols = spectrum_qam_symbols(spectrum_bits, order)
    baseband = np.repeat(symbols, samples_per_symbol)
    spectrum_time = np.arange(len(baseband)) / spectrum_fs
    for carrier in spectrum_carriers:
        wave = np.real(baseband * np.exp(2j * np.pi * carrier * spectrum_time))
        frequencies, psd = welch(
            wave, fs=spectrum_fs, window="hann", nperseg=4096,
            noverlap=2048, detrend=False, scaling="density",
        )
        # Normalize each spectrum to its own peak to compare widths, not heights.
        relative_psd = psd / psd.max()
        spectrum_results.append(dict(
            label=label, color=color, carrier=carrier, symbol_rate=symbol_rate,
            frequencies=frequencies,
            level_db=10 * np.log10(np.maximum(relative_psd, 1e-8)),
        ))

print(f"Shared message: {len(spectrum_bits)} bits, "
      f"{len(spectrum_bits) / spectrum_bit_rate:.2f} s at {spectrum_bit_rate} bit/s")
for order, label, _ in spectrum_schemes:
    rs = spectrum_bit_rate / np.log2(order)
    print(f"{label}: {rs:.0f} symbols/s; main-lobe width {2 * rs:.0f} Hz")


**Compare carrier positions and main-lobe widths**


In [ ]:
fig = plt.figure(figsize=(12, 8), layout="constrained")
grid = fig.add_gridspec(2, 2, height_ratios=[1, 1.25])
overview = fig.add_subplot(grid[0, :])
zoom_axes = [fig.add_subplot(grid[1, column]) for column in range(2)]

for result in spectrum_results:
    overview.plot(
        result["frequencies"], result["level_db"], color=result["color"],
        linewidth=1.1, alpha=0.8,
        linestyle="-" if result["carrier"] == spectrum_carriers[0] else "--",
        label=f'{result["label"]}, carrier {result["carrier"]:.0f} Hz',
    )
for carrier in spectrum_carriers:
    overview.axvline(carrier, color="0.45", linewidth=0.8, alpha=0.5)
overview.set(
    xlim=(0, max(spectrum_carriers) + 400), ylim=(-55, 4),
    xlabel="Frequency (Hz)", ylabel="Relative PSD (dB)",
    title="Carrier positions: averaged spectra at the same bit rate",
)
overview.legend(fontsize=8, ncol=2, loc="lower right")
overview.grid(alpha=0.2)

for axis, carrier in zip(zoom_axes, spectrum_carriers):
    for result in spectrum_results:
        if result["carrier"] != carrier:
            continue
        rs, color = result["symbol_rate"], result["color"]
        axis.plot(result["frequencies"], result["level_db"],
                  color=color, linewidth=1, alpha=0.45)
        # Expected positive-frequency lobe for rectangular symbol pulses.
        envelope_f = np.linspace(max(0, carrier - 250), carrier + 250, 3001)
        envelope = np.sinc((envelope_f - carrier) / rs) ** 2
        axis.plot(envelope_f, 10 * np.log10(np.maximum(envelope, 1e-8)),
                  color=color, linewidth=2, linestyle="--",
                  label=f'{result["label"]}: {rs:.0f} symbols/s')
        axis.axvspan(carrier - rs, carrier + rs, color=color, alpha=0.07)
        arrow_y = -41 if rs == 200 else -49
        axis.annotate("", xy=(carrier - rs, arrow_y),
                      xytext=(carrier + rs, arrow_y),
                      arrowprops=dict(arrowstyle="<->", color=color, linewidth=1.6))
        axis.text(carrier, arrow_y + 1.5, f"{2 * rs:.0f} Hz main lobe",
                  ha="center", va="bottom", color=color, fontsize=9,
                  bbox=dict(facecolor="white", edgecolor="none", alpha=0.8, pad=1))
    axis.set(
        xlim=(carrier - 250, carrier + 250), ylim=(-55, 4),
        xlabel="Frequency (Hz)", ylabel="Relative PSD (dB)",
        title=f"Bandwidth comparison around {carrier:.0f} Hz",
    )
    axis.legend(fontsize=8, loc="upper right")
    axis.grid(alpha=0.2)

fig.suptitle(
    "QPSK versus 256-QAM at 400 bit/s: 400 Hz versus 100 Hz main-lobe width\n"
    "Lower panels: thin lines show averaged spectra; dashed curves show theoretical envelopes",
    fontsize=12,
)
plt.show()


### 8.5 Spectral Efficiency and Radio Spectrum Regulation

The spectrum comparison demonstrates two distinct design choices. Changing the carrier frequency moves a signal to a different part of the spectrum. Changing the symbol rate changes its spectral width when the pulse shape remains fixed. Higher-order modulation can carry a fixed bit rate with fewer symbols per second, or carry more bits per second at the same symbol rate.

**Spectral efficiency** is the ratio of data rate to bandwidth:

$$
\eta = \frac{R_b}{B}\qquad [\mathrm{bit/s/Hz}].
$$

The comparison must state which data rate and bandwidth are being used. In this notebook, $R_b$ counts the modulation bits and the marked bandwidth is the main-lobe width between the first zeros. Coding, protocol overhead, pulse shaping, and a different occupied-bandwidth criterion can change the efficiency reported for a practical system. Bits per symbol alone do not specify spectral efficiency.

For users, efficient spectrum use can support higher data rates when the channel is sufficiently clean. For a network operator, it can increase the data carried within an existing frequency allocation or reduce the bandwidth needed for a given service. For spectrum management, it can help accommodate demand while preserving room for other services. These are potential benefits under suitable operating conditions, not automatic consequences of choosing a larger constellation.

Reliable coexistence also requires control of transmitted power, unwanted emissions, and receiver filtering, together with appropriate coordination between users. A narrower main lobe does not eliminate sidelobes or guarantee interference-free operation. The [ITU handbook on national spectrum management](https://www.itu.int/dms_pub/itu-r/opb/hdb/R-HDB-21-2005-R1-PDF-E.pdf) describes the broader coordination needed to use spectrum efficiently and limit harmful interference.

The practical choice balances data rate, bandwidth, and reception quality. Higher-order QAM is useful when the receiver can distinguish its closely spaced states. Under noisier conditions, a lower modulation order may deliver data more reliably. Spectrum efficiency therefore has value only when the link also meets its communication requirements.


----

### 8.6 Hands-On Exercises

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>

**What is a baseband signal, and what does the term “baseband” indicate about its frequency content?**

(answer here)

**Why do conventional mobile radio systems usually shift a baseband signal onto a carrier instead of transmitting it directly? Explain the roles of antenna size, wavelength, and separation between frequency channels. How does this differ from transmission over a suitable wired link?**

(answer here)

**In Section 8.4, both QPSK and 256-QAM transmit at 400 bit/s. Use their bits per symbol and symbol rates to explain why the QPSK main lobe is four times wider. What bandwidth measure is marked by the arrows in the graph?**

(answer here)

**Why does a radio system not always use the highest modulation order available? Compare QPSK and 256-QAM at the same average signal power and explain how noise affects the choice.**

(answer here)

**Define spectral efficiency and state its units. Explain one potential benefit of higher spectral efficiency for each of the following: users, network operators, and spectrum management. Why does higher spectral efficiency alone not guarantee interference-free operation?**

(answer here)




**OPTIONAL: Did you read through and understand all the tasks in this Notebook? Are there any shortcomings, missing parts, or sections that should be improved?**

 (please help the team by telling us what part was not understood, and we will work on it)